In [2]:
import pandas as pd

In [3]:
begin_inventory = pd.read_csv("../data/raw/begin_inventory.csv")
end_inventory = pd.read_csv("../data/raw/end_inventory.csv")
purchase_prices = pd.read_csv("../data/raw/purchase_prices.csv")
purchases = pd.read_csv("../data/raw/purchases.csv")
sales = pd.read_csv("../data/raw/sales.csv")
vendor_invoice = pd.read_csv("../data/raw/vendor_invoice.csv")
vendor_sales_summary = pd.read_csv("../data/raw/vendor_sales_summary.csv")

 Standardize Columns Names

In [4]:
# # sales.csv uses "VendorNo" but every other file uses "VendorNumber"
sales = sales.rename(columns = {"VendorNo": "VendorNumber"})
sales.columns.tolist()

['InventoryId',
 'Store',
 'Brand',
 'Description',
 'Size',
 'SalesQuantity',
 'SalesDollars',
 'SalesPrice',
 'SalesDate',
 'Volume',
 'Classification',
 'ExciseTax',
 'VendorNumber',
 'VendorName']

In [5]:
#columns of every file side by side to manually verify


print("begin_inventory:", begin_inventory.columns.tolist())
print("end_inventory:", end_inventory.columns.tolist())
print("purchase_prices:", purchase_prices.columns.tolist())
print("purchases:", purchases.columns.tolist())
print("sales:", sales.columns.tolist())
print("vendor_invoice:", vendor_invoice.columns.tolist())
print("vendor_sales_summary:", vendor_sales_summary.columns.tolist())

begin_inventory: ['InventoryId', 'Store', 'City', 'Brand', 'Description', 'Size', 'onHand', 'Price', 'startDate']
end_inventory: ['InventoryId', 'Store', 'City', 'Brand', 'Description', 'Size', 'onHand', 'Price', 'endDate']
purchase_prices: ['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification', 'PurchasePrice', 'VendorNumber', 'VendorName']
purchases: ['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber', 'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate', 'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification']
sales: ['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity', 'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification', 'ExciseTax', 'VendorNumber', 'VendorName']
vendor_invoice: ['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate', 'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval']
vendor_sales_summary: ['VendorNumber', 'VendorName', 'Brand', 'Description', 'Purchas

Fix Datatypes

In [6]:
# begin_inventory and end_inventory have one date column each
begin_inventory["startDate"] = pd.to_datetime(begin_inventory["startDate"])
end_inventory["endDate"] = pd.to_datetime(end_inventory["endDate"])

#purchase has 4 date columns
purchases["PODate"] = pd.to_datetime(purchases["PODate"])
purchases["ReceivingDate"] = pd.to_datetime(purchases["ReceivingDate"])
purchases["InvoiceDate"] = pd.to_datetime(purchases["InvoiceDate"])
purchases["PayDate"] = pd.to_datetime(purchases["PayDate"])

#sales has 1 date column
sales["SalesDate"] = pd.to_datetime(sales["SalesDate"])

# vendor_invoice has 3 date columns
vendor_invoice["InvoiceDate"] = pd.to_datetime(vendor_invoice["InvoiceDate"])
vendor_invoice["PODate"] = pd.to_datetime(vendor_invoice["PODate"])
vendor_invoice["PayDate"] = pd.to_datetime(vendor_invoice["PayDate"])

In [7]:
# dtype should now say "datetime64" instead of "object" for these columns
print(begin_inventory[["startDate"]].dtypes)
print(end_inventory[["endDate"]].dtypes)
print(purchases[["PODate", "ReceivingDate", "InvoiceDate", "PayDate"]].dtypes)
print(sales[["SalesDate"]].dtypes)
print(vendor_invoice[["InvoiceDate", "PODate", "PayDate"]].dtypes)


startDate    datetime64[us]
dtype: object
endDate    datetime64[us]
dtype: object
PODate           datetime64[us]
ReceivingDate    datetime64[us]
InvoiceDate      datetime64[us]
PayDate          datetime64[us]
dtype: object
SalesDate    datetime64[us]
dtype: object
InvoiceDate    datetime64[us]
PODate         datetime64[us]
PayDate        datetime64[us]
dtype: object


In [8]:
print(purchase_prices["Volume"].unique())

<ArrowStringArray>
[    '750',    '1000',    '1750',      '50',     '375',     '100',     '200',
     '300', 'Unknown',     '250',    '1500',    '3000',    '5000',    '4000',
     '187',     '150',     '500',     '720',     '650',     '330',   '18000',
     '180',    '6000',      '20',       nan,   '20000',   '162.5',     '400',
    '1100',     '600',   '19500',     '560',    '3750',    '9000']
Length: 34, dtype: str


In [9]:
print(purchase_prices["Volume"].value_counts().tail(10))
#Specifically count "Unkonown"
print("\nRows with 'Unkown : ", (purchase_prices["Volume"] =="Unknown").sum())

Volume
600      2
560      2
650      1
20       1
162.5    1
400      1
1100     1
19500    1
3750     1
9000     1
Name: count, dtype: int64

Rows with 'Unkown :  4


In [10]:
unknown_rows = purchase_prices[purchase_prices["Volume"] == "Unknown"]
print(unknown_rows[["Brand","Description","Size", "Volume"]])

       Brand                   Description     Size   Volume
542     2993             Angostura Bitters  Unknown  Unknown
5921    9908      Tito's Copper Mug 2 Pack  Unknown  Unknown
8795    8992                      Group 92  Unknown  Unknown
10009  90590  Overture Champagne 2Glass Pk  Unknown  Unknown


In [11]:
purchase_prices["Volume"] = pd.to_numeric(purchase_prices["Volume"], errors = "coerce")
print(purchase_prices["Volume"].dtype)
print("Missing Volume values:", purchase_prices["Volume"].isnull().sum())

float64
Missing Volume values: 5


Drop rows with missing volume

In [12]:
print("Before drop:", purchase_prices.shape)

purchase_prices = purchase_prices.dropna(subset=["Volume"])

print("After drop", purchase_prices.shape)
print("Missing Volume values now:", purchase_prices["Volume"].isnull().sum())

Before drop: (12261, 9)
After drop (12256, 9)
Missing Volume values now: 0


Investigate Approval Column

In [13]:
# what are the actual values when approval IS filled in?
print(vendor_invoice["Approval"].value_counts())

Approval
Frank Delahunt    374
Name: count, dtype: int64


In [14]:
# Does an invoice getting flagged correlate with unusually high $ amounts?
# This tells us if Approval behaves like a "risk flag"
print(vendor_invoice.groupby(vendor_invoice["Approval"].isnull())[["Quantity", "Dollars", "Freight"]].mean())

              Quantity        Dollars      Freight
Approval                                          
False     48172.927807  484859.703663  2420.527193
True       3011.743471   27193.506744   142.232060


convert Approval to a 0/1 flag

In [15]:
#NAN means "not flagged "  --> 0
#Any actual value means "Flagged" --> 1

vendor_invoice["Approval_Flag"] = vendor_invoice["Approval"].notnull().astype(int)
print(vendor_invoice["Approval_Flag"].value_counts())

Approval_Flag
0    5169
1     374
Name: count, dtype: int64


### Investigate missing City in end_inventory

In [16]:
# Check if these rows have a valid Store number we can map back from begin_inventory
missing_city_rows = end_inventory[end_inventory["City"].isnull()]
print("Missing City Rows:",missing_city_rows.shape[0])
print(missing_city_rows["Store"].unique())

Missing City Rows: 1284
[46]


In [17]:
# If yes, we can fill the missing City using this mapping instead of dropping data
store_city_map = begin_inventory[["Store", "City"]].drop_duplicates()
missing_stores = missing_city_rows["Store"].unique()
print(store_city_map[store_city_map["Store"].isin(missing_stores)])


        Store         City
108343     46  TYWARDREATH


In [18]:
# We know Store 46 = TYWARDREATH from begin_inventory
end_inventory["City"] = end_inventory["City"].fillna("TYWARDREATH")
print("Missing City now:", end_inventory["City"].isnull().sum())

Missing City now: 0


In [19]:
# duplicte check
print("begin_inventory duplicates:", begin_inventory.duplicated().sum())
print("end_inventory duplicates:", end_inventory.duplicated().sum())
print("purchase_prices duplicates:", purchase_prices.duplicated().sum())
print("purchases duplicates:", purchases.duplicated().sum())
print("sales duplicates:", sales.duplicated().sum())
print("vendor_invoice duplicates:", vendor_invoice.duplicated().sum())

begin_inventory duplicates: 0
end_inventory duplicates: 0
purchase_prices duplicates: 0
purchases duplicates: 0
sales duplicates: 0
vendor_invoice duplicates: 0


In [20]:
# extreme values in purchases.Quantity
print(purchases["Quantity"].describe())

#Look at top 5 largest purchase quantities
print(purchases.sort_values("Quantity", ascending = False).head(5)[["Brand", "Description","VendorName","Quantity","Dollars"]])

count    2.372474e+06
mean     1.415585e+01
std      2.344616e+01
min      1.000000e+00
25%      6.000000e+00
50%      1.000000e+01
75%      1.200000e+01
max      3.816000e+03
Name: Quantity, dtype: float64
         Brand           Description                   VendorName  Quantity  \
2130897   5609         Integre Vodka  M S WALKER INC                   3816   
1661600   1376              Jim Beam  JIM BEAM BRANDS COMPANY          2425   
1194992   4246  Bacardi Superior Rum  BACARDI USA INC                  2159   
1194496   3858      Grey Goose Vodka  BACARDI USA INC                  2055   
2130168   5614  Integre Citrus Vodka  M S WALKER INC                   1920   

          Dollars  
2130897  17858.88  
1661600  39503.25  
1194992   9110.98  
1194496  36517.35  
2130168   7660.80  


In [21]:
print(purchases["Dollars"].describe())
print(sales["SalesQuantity"].describe())

count    2.372474e+06
mean     1.356815e+02
std      2.816649e+02
min      0.000000e+00
25%      4.926000e+01
50%      8.393000e+01
75%      1.405200e+02
max      5.017570e+04
Name: Dollars, dtype: float64
count    1.282536e+07
mean     2.566623e+00
std      4.551810e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      1.231000e+03
Name: SalesQuantity, dtype: float64


## SQLite db

In [27]:
import sqlite3

conn = sqlite3.connect("../database/vendor_invoice.db")

begin_inventory.to_sql("begin_inventory", conn, if_exists="replace", index = False)
end_inventory.to_sql("end_inventory", conn, if_exists="replace", index = False)
purchase_prices.to_sql("purchase_prices",  conn, if_exists = "replace", index = False)

# Large files - writes 50000 rows at a time
purchases.to_sql("purchases", conn, if_exists="replace", index = False, chunksize = 50000 )
sales.to_sql("sales", conn, if_exists = "replace", index= False, chunksize = 50000)
vendor_invoice.to_sql("vendor_invoice", conn, if_exists="replace", index = False)
vendor_sales_summary.to_sql("vendor_sales_summary", conn, if_exists = "replace", index= False)

conn.close()
print("All tables saved to SQLite")

All tables saved to SQLite


Save all files as processed csv

In [25]:
begin_inventory.to_csv("../data/processed/begin_inventory_clean.csv", index = False)
end_inventory.to_csv("../data/processed/end_inventory_clean.csv", index = False)
purchase_prices.to_csv("../data/processed/purchase_prices_clean.csv", index = False)
purchases.to_csv("../data/processed/purchases_clean.csv", index = False)
vendor_invoice.to_csv("../data/processed/vendor_invoice.csv", index = False)
vendor_sales_summary.to_csv("../data/processed/vendor_sales_summary.csv", index = False)

print("Processed CSVs saved")

Processed CSVs saved


In [28]:
conn = sqlite3.connect("../database/vendor_invoice.db")

tables = ["begin_inventory", "end_inventory", "purchase_prices", "purchases", "sales","vendor_invoice", "vendor_sales_summary"]
for table in tables:
    count = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {tables}", conn)
    print(f"{table}:{count['cnt'][0] }rows")

conn.close()



DatabaseError: Execution failed on sql 'SELECT COUNT(*) as cnt FROM ['begin_inventory', 'end_inventory', 'purchase_prices', 'purchases', 'sales', 'vendor_invoice', 'vendor_sales_summary']': no such table: 'begin_inventory', 'end_inventory', 'purchase_prices', 'purchases', 'sales', 'vendor_invoice', 'vendor_sales_summary'